# Spatial relationships in Calgary's water network

How closely do historical breaks align with the current pipe inventory? How much pipe intersects each 500 m sector, and where have breaks recurred?

Run the cells in order from a HYDROSIGHT checkout. In Colab, use the checkout at `/content/HYDROSIGHT`. The inputs are the latest materialized clean exports in `data/processed/pipes/` and `data/processed/water_breaks/` and the official municipal boundary of Calgary. The notebook builds its 500 m sectors in memory. Run all cells to review the spatial diagnostics and export the sector summary.

In [ ]:
from pathlib import Path
from datetime import datetime, timezone
import json
import sys

import geopandas as gpd
import numpy as np
import pandas as pd
from shapely.geometry import shape

candidates = [Path.cwd(), *Path.cwd().parents, Path("/content/HYDROSIGHT")]
root = next((p for p in candidates if (p / "src/hydrosight/preprocessing/break_history.py").is_file()), None)
if root is None:
    raise FileNotFoundError("Open this notebook in the HYDROSIGHT checkout.")

sys.path.insert(0, str(root / "src"))
from hydrosight.preprocessing.break_history import aggregate_break_history, assign_breaks_to_grid

data = root / "data"
processed = data / "processed"

def clean_input(dataset, prefixes):
    folder = processed / dataset
    snapshots = [
        path for prefix in prefixes for path in folder.glob(f"{prefix}_*.jsonl")
        if path.is_file()
    ]
    if snapshots:
        return max(snapshots, key=lambda path: (path.stem.rsplit("_", 1)[-1], path.name))
    raise FileNotFoundError(f"No clean {dataset} export in {folder}. Run its source EDA notebook first.")


pipes_path = clean_input("pipes", ["public_water_mains_clean", "public_water_means_clean"])
breaks_path = clean_input("water_breaks", ["water_main_breaks_clean"])
df_clean = pd.read_json(pipes_path, lines=True)
breaks = pd.read_json(breaks_path, lines=True)
for path, records in [(pipes_path, df_clean), (breaks_path, breaks)]:
    print(f"{path.relative_to(root)}: {len(records):,} records")

In [ ]:
from hashlib import sha256
from io import BytesIO
from urllib.request import urlopen

from shapely import set_precision
from shapely.geometry import box

boundary_url = (
    "https://services1.arcgis.com/AVP60cs0Q9PEA8rH/arcgis/rest/services/"
    "City_Boundary/FeatureServer/0/query?where=1%3D1&outFields=*&outSR=4326"
    "&returnGeometry=true&f=geojson"
)
boundary_path = processed / "calgary_boundary.geojson"
if boundary_path.is_file():
    boundary = gpd.read_file(boundary_path)
else:
    with urlopen(boundary_url, timeout=60) as response:
        boundary = gpd.read_file(BytesIO(response.read()))

if (boundary.empty or boundary.crs is None or not boundary.geometry.is_valid.all()
        or not boundary.geom_type.isin(["Polygon", "MultiPolygon"]).all()):
    raise ValueError("A valid georeferenced municipal boundary is required.")

cell_size_m = 500
metric_crs = "EPSG:32611"
city = boundary.to_crs(metric_crs).geometry.union_all()
min_x, min_y, max_x, max_y = city.bounds
sectors = []
for ix in range(int(np.floor(min_x / cell_size_m)), int(np.ceil(max_x / cell_size_m))):
    for iy in range(int(np.floor(min_y / cell_size_m)), int(np.ceil(max_y / cell_size_m))):
        x, y = ix * cell_size_m, iy * cell_size_m
        clipped = box(x, y, x + cell_size_m, y + cell_size_m).intersection(city)
        if clipped.area > 0:
            sectors.append({"cell_id": f"CELL_{x}_{y}", "geometry": clipped})
grid = gpd.GeoDataFrame(sectors, geometry="geometry", crs=metric_crs).to_crs("EPSG:4326")

# Identify the boundary and construction rules used for this export.
fingerprint = sha256(set_precision(city, 0.00001).normalize().wkb).hexdigest()[:16]
grid_definition = f"{fingerprint}|{cell_size_m}|{metric_crs}|clipped-positive-area-v1"
grid_version_id = sha256(grid_definition.encode()).hexdigest()[:16]
print(f"Grid version: {grid_version_id} | Sectors: {len(grid):,}")

## 1. Historical breaks and nearby pipes

For each break, find the nearest mapped pipe within 30 m. An ambiguous match means multiple pipes tied for the shortest distance. Distances describe proximity to the current network; a unique nearest pipe alone cannot identify which pipe failed.

In [ ]:
def assign_pipes_to_grid(
    df_pipes: pd.DataFrame,
    grid: gpd.GeoDataFrame
) -> gpd.GeoDataFrame:
    """Assign pipes to spatial grid cells using their geometries."""

    pipes = gpd.GeoDataFrame(
        df_pipes.copy(),
        geometry=df_pipes["multilinestring"].apply(
            lambda value: shape(json.loads(value))
            if isinstance(value, str)
            else shape(value)
        ),
        crs="EPSG:4326"
    )

    pipes = pipes.to_crs(grid.crs)

    gdf_pipes_grid = gpd.sjoin(
        pipes,
        grid[["cell_id", "geometry"]],
        how="left",
        predicate="intersects"
    )

    return gdf_pipes_grid


def match_breaks_to_pipes(
    gdf_pipes: gpd.GeoDataFrame,
    gdf_water_breaks: gpd.GeoDataFrame,
) -> gpd.GeoDataFrame:
    """Match water main breaks to nearby pipes using metric distances."""

    pipes = (
        gdf_pipes
        .drop(columns=["index_right"], errors="ignore")
        .drop_duplicates(subset=["globalid"])
        .to_crs("EPSG:32611")
    )

    breaks = (
        gdf_water_breaks
        .drop(columns=["index_right"], errors="ignore")
        .reset_index(drop=True)
        .to_crs("EPSG:32611")
        .copy()
    )

    breaks["break_id"] = breaks.index

    gdf_pipe_breaks = gpd.sjoin_nearest(
        breaks,
        pipes[["globalid", "geometry"]],
        how="left",
        max_distance=30,
        distance_col="distance_m",
    )

    gdf_pipe_breaks = gdf_pipe_breaks.drop(
        columns=["index_right"],
        errors="ignore"
    )

    gdf_pipe_breaks["match"] = np.select(
        [
            gdf_pipe_breaks["globalid"].isna(),
            gdf_pipe_breaks.duplicated(
                subset=["break_id"],
                keep=False
            )
        ],
        [
            "Absent",
            "Ambiguous"
        ],
        default="Unique"
    )

    return gdf_pipe_breaks


def print_pipe_match_summary(
    gdf_pipe_breaks: gpd.GeoDataFrame,
    original_break_count: int,
) -> None:
    break_match_summary = gdf_pipe_breaks.drop_duplicates(subset=["break_id"])

    print("=== Breaks and the nearest pipes in the current network ===")
    print(f"Total breaks in the full history: {original_break_count:,}")
    print("Match status (one count per break):")
    match_counts = break_match_summary["match"].value_counts().rename_axis(None)
    print(match_counts.to_string())

    print("\nDistances to candidate pipes (m, one row per candidate):")
    print(gdf_pipe_breaks["distance_m"].describe().to_string())

In [ ]:
pipes_by_sector = assign_pipes_to_grid(df_clean, grid)
breaks_by_sector = assign_breaks_to_grid(breaks, grid)
matches = match_breaks_to_pipes(pipes_by_sector, breaks_by_sector)
print_pipe_match_summary(matches, len(breaks))

## 2. Current pipe coverage by sector

Summarize pipe–sector intersections and the length of pipe inside each sector. The project's intersection rule includes boundary touches. A sector with an intersection can therefore have zero measured pipe length.

In [ ]:
def aggregate_pipe_features(
    gdf_pipes_grid: gpd.GeoDataFrame,
    grid: gpd.GeoDataFrame,
) -> pd.DataFrame:
    """Aggregate pipe characteristics by spatial grid cell."""

    pipes = gdf_pipes_grid.dropna(subset=["cell_id"]).copy()

    pipes = pipes.to_crs("EPSG:32611")
    cells = grid.to_crs("EPSG:32611")

    cell_geometries = cells.set_index("cell_id")["geometry"]

    pipes["pipe_length_in_cell_m"] = pipes.apply(
        lambda row: row.geometry.intersection(
            cell_geometries.loc[row["cell_id"]]
        ).length,
        axis=1,
    )

    df_features = (
        pipes.groupby("cell_id")
        .agg(
            pipe_count=("globalid", "nunique"),
            pipe_length_m=("pipe_length_in_cell_m", "sum"),
            avg_diameter=("diam", "mean"),
            material_count=("material", "nunique"),
        )
        .reset_index()
    )

    return df_features


def print_pipe_coverage_summary(
    gdf_pipes_grid: gpd.GeoDataFrame,
    df_pipe_features: pd.DataFrame,
) -> None:
    duplicate_pairs = gdf_pipes_grid.duplicated(subset=["globalid", "cell_id"]).sum()
    total_length_m = round(df_pipe_features["pipe_length_m"].sum(), 2)

    print("\n=== Current pipe network by sector ===")
    print(f"Sectors with pipes: {len(df_pipe_features):,}")
    print(f"Total network length in the grid: {total_length_m:,.2f} m")
    print(f"Duplicate pipe–sector pairs: {duplicate_pairs:,}")

    print("\nSample of pipe features by sector:")
    print(df_pipe_features.head().to_string(index=False))

In [ ]:
pipe_features = aggregate_pipe_features(pipes_by_sector, grid)
print_pipe_coverage_summary(pipes_by_sector, pipe_features)
print("Sectors with positive pipe length:", pipe_features["pipe_length_m"].gt(0).sum())

## 3. Recorded break recurrence and network overlap, 2016–2025

Use the monthly sector history for the selected break snapshot during the ten complete calendar years in this comparison. Compare sectors with observed breaks against sectors intersecting the current pipe inventory. A sector with no recorded break may have incomplete observation coverage. The current inventory also may differ from the network present when the break occurred.

In [ ]:
def print_break_coverage_and_recurrence(
    df_break_history: pd.DataFrame,
    breaks_in_period: pd.DataFrame,
    pipe_cells: set[str],
) -> None:
    break_cells = set(breaks_in_period["cell_id"].dropna())

    print("\n=== Recorded breaks by sector ===")
    print(f"Breaks assigned to the grid, all years: {df_break_history['break_count'].sum():,}")
    print("\nSample of monthly break counts from the full history:")
    print(df_break_history.head().to_string(index=False))

    print("\nBreaks in 2016–2025 and overlap with the current pipe network:")
    print(f"  Recorded breaks: {breaks_in_period['break_count'].sum():,}")
    print(f"  Sectors with recorded breaks: {len(break_cells):,}")
    print(f"  Sectors with current pipes and recorded breaks: {len(pipe_cells & break_cells):,}")
    print(f"  Sectors with recorded breaks and no current pipes: {len(break_cells - pipe_cells):,}")
    print(f"  Sectors with current pipes and no recorded breaks: {len(pipe_cells - break_cells):,}")

    breaks_per_cell = breaks_in_period.groupby("cell_id")["break_count"].sum()

    print("\nBreak recurrence by sector (2016–2025):")
    print(f"  Exactly one break: {breaks_per_cell.eq(1).sum():,}")
    print(f"  Two or more breaks: {breaks_per_cell.ge(2).sum():,}")
    print(f"  Highest break count in one sector: {breaks_per_cell.max():,}")

In [ ]:
monthly = aggregate_break_history(breaks_by_sector)

if monthly["break_count"].sum() != breaks_by_sector["cell_id"].notna().sum():
    raise ValueError("Monthly totals differ from the number of assigned breaks.")

study_months = monthly.loc[monthly["month"].dt.year.between(2016, 2025)]
pipe_cells = set(pipe_features["cell_id"].dropna())
print_break_coverage_and_recurrence(monthly, study_months, pipe_cells)

## 4. Review the sector results

Review one row per active grid sector. Pipe measures describe the current inventory. The observed break count covers 2016–2025; zero means the source contains no recorded break for that sector during the period. Nearest-pipe matches have a different unit of analysis and remain in the event-level diagnostic above.

In [ ]:
observed = study_months.groupby("cell_id")["break_count"].sum().rename("observed_breaks_2016_2025")
observed = observed.reindex(grid["cell_id"], fill_value=0)
sector_review = (
    grid[["cell_id"]].merge(pipe_features, on="cell_id", how="left", validate="one_to_one")
    .merge(observed, on="cell_id", how="left", validate="one_to_one")
    .sort_values("cell_id")
)
print(sector_review.drop(columns="cell_id").describe().T)
sector_review.head(10)

## 5. Decisions and sector export

Retain one row for every cell in the active grid. Export current mapped pipe length in metres and the observed break count for 2016–2025. A cell without a pipe intersection has zero mapped length in this inventory. A cell with no recorded breaks has a count of zero. Using a sector as a negative training example requires separate evidence of observation coverage. Nearest-pipe matches remain exploratory. The filenames and grid version identify the inputs used.

In [ ]:
result = sector_review[["cell_id", "pipe_length_m", "observed_breaks_2016_2025"]].copy()
without_mapped_pipes = ~result["cell_id"].isin(pipe_cells)
result.loc[without_mapped_pipes, "pipe_length_m"] = 0.0
result["grid_version"] = grid_version_id
result["pipe_snapshot"] = pipes_path.name
result["break_snapshot"] = breaks_path.name

output_dir = processed / "spatial_relationships"
output_dir.mkdir(parents=True, exist_ok=True)
timestamp = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S%fZ")
output_file = output_dir / f"spatial_relationships_clean_{timestamp}.jsonl"
result.to_json(output_file, orient="records", lines=True, force_ascii=False)
print(f"Saved {len(result):,} sectors to {output_file}")